# Task-Conditioned VAE — Named Latent Space for RF Pulse Signals

**Architecture**: 1D-CNN encoder → named latent space → bespoke per-parameter decoders  
**Latent slots**: `pulse_duration · mod_type · mod_content · filter_shape · rise_time · fall_time · amplitude · residuals`  
**Key techniques**: Gumbel-softmax (discrete mod_type), Complex-Valued NN (filter_shape), KL + auxiliary loss warmup  
**Dataset**: RadioML 2016.10A (auto-downloaded, ~55 MB) or built-in synthetic generator

See `README.md` for paper references and architectural rationale.

## 0. Config

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import windows
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from dataclasses import dataclass, field
from typing import Optional
import os, pickle, urllib.request, hashlib

torch.manual_seed(42)
np.random.seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── Dataset choice ────────────────────────────────────────────────────────────
# Set True to download and use RadioML 2016.10A (~55 MB subset, free)
# Set False to use the built-in synthetic generator (no download needed)
USE_RADIOML = True
RADIOML_PATH = './radioml_2016_10a_subset.pkl'   # cached after first download

# ── Signal parameters ─────────────────────────────────────────────────────────
N_SAMPLES     = 128      # IQ samples per burst (RadioML native length)

# RadioML 2016.10A classes we map onto our 4-class taxonomy
# (others are routed to 'unknown' and handled by residuals slot)
RADIOML_CLASS_MAP = {
    'AM-SSB': 0,   # → CW-like (narrowband AM)
    'WBFM':   0,   # → CW-like (wideband FM)
    'AM-DSB': 0,   # → CW-like
    'CPFSK':  1,   # → LFM-like (frequency-shift keying, continuous phase)
    'GFSK':   1,   # → LFM-like (Gaussian FSK)
    'BPSK':   2,   # → BPSK
    'QPSK':   3,   # → QPSK
    '8PSK':   3,   # → QPSK-like (higher order PSK)
    'QAM16':  3,   # → QPSK-like (QAM maps to phase modulation bucket)
    'QAM64':  3,   # → QPSK-like
    'PAM4':   2,   # → BPSK-like (pulse amplitude mod)
}
N_MOD_CLASSES = 4
MOD_NAMES     = ['CW/AM', 'LFM/FSK', 'BPSK/PAM', 'QPSK/QAM']

# ── Latent dimensions ────────────────────────────────────────────────────────
DIM_PULSE_DUR  = 1
DIM_MOD_TYPE   = N_MOD_CLASSES
DIM_MOD_CONT   = 8
DIM_FILTER     = 8
DIM_RISE       = 1
DIM_FALL       = 1
DIM_AMPLITUDE  = 1
DIM_RESIDUALS  = 4
DIM_TOTAL = (DIM_PULSE_DUR + DIM_MOD_TYPE + DIM_MOD_CONT +
             DIM_FILTER + DIM_RISE + DIM_FALL + DIM_AMPLITUDE + DIM_RESIDUALS)
print(f'Total latent dim: {DIM_TOTAL}')

# ── Training config ───────────────────────────────────────────────────────────
BATCH_SIZE    = 128
N_EPOCHS      = 60
LR            = 3e-4
BETA_MAX      = 0.4
WARMUP_EPOCHS = 15
AUX_START     = 10
SNR_FILTER_DB = 10    # for RadioML: only use bursts at SNR >= this value (cleaner training signal)
N_SYNTHETIC   = 8000  # samples if using synthetic generator

## 1. Dataset — RadioML 2016.10A or synthetic

In [ ]:
# ── RadioML 2016.10A loader ───────────────────────────────────────────────────
# Dataset: O'Shea & West, 'Radio Machine Learning Dataset Generation with GNU Radio' (2016)
# 220,000 bursts × 128 IQ samples, 11 modulation classes, SNR −20..+18 dB
# The subset URL below (~55 MB) is a publicly mirrored pickle of the first 20k samples
# at SNR ≥ 0 dB, sufficient for PoC.

RADIOML_SUBSET_URL = (
    'https://huggingface.co/datasets/Jakaria816/RadioML-2016/resolve/main/'
    'RML2016.10a_dict.pkl'
)

def download_radioml(dest: str) -> bool:
    """Download RadioML 2016.10A pickle. Returns True on success."""
    if os.path.exists(dest):
        print(f'  Found cached RadioML at {dest}')
        return True
    print(f'  Downloading RadioML 2016.10A …')
    try:
        urllib.request.urlretrieve(RADIOML_SUBSET_URL, dest,
            reporthook=lambda b, bs, t: print(
                f'  {min(100, int(b*bs/max(1,t)*100)):3d}%', end='\r') if t > 0 else None)
        print('  Download complete.')
        return True
    except Exception as e:
        print(f'  Download failed: {e}')
        if os.path.exists(dest): os.remove(dest)
        return False


def load_radioml(path: str, snr_min: int = SNR_FILTER_DB):
    """
    Load RadioML 2016.10A pickle.
    Returns (iq_array, mod_labels, snr_labels) all as numpy arrays.
    iq_array: (N, 128) interleaved I&Q float32
    """
    with open(path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    iq_list, mod_list, snr_list = [], [], []
    for (mod_name, snr), samples in data.items():
        if snr < snr_min: continue
        if mod_name not in RADIOML_CLASS_MAP: continue
        cls = RADIOML_CLASS_MAP[mod_name]
        for s in samples:
            # s shape: (2, 128) — row 0 = I, row 1 = Q
            # Interleave to (128,): [I0, Q0, I1, Q1, ...]
            iq = np.empty(128, dtype=np.float32)
            iq[0::2] = s[0]   # I
            iq[1::2] = s[1]   # Q
            iq_list.append(iq)
            mod_list.append(cls)
            snr_list.append(float(snr))

    iq_arr  = np.stack(iq_list).astype(np.float32)
    mod_arr = np.array(mod_list, dtype=np.int64)
    snr_arr = np.array(snr_list, dtype=np.float32)

    # Normalise per-sample to unit max amplitude
    scale = np.abs(iq_arr).max(axis=1, keepdims=True).clip(min=1e-8)
    iq_arr = iq_arr / scale

    print(f'  Loaded {len(iq_arr):,} bursts | classes: {dict(zip(*np.unique(mod_arr, return_counts=True)))}')
    return iq_arr, mod_arr, snr_arr

In [ ]:
# ── Synthetic generator (fallback / augmentation) ─────────────────────────────

def sinc_filter(n: int, bw: float) -> np.ndarray:
    t = np.arange(-(n//2), n//2 + (n % 2))
    h = np.sinc(2 * bw * t) * windows.hann(len(t))
    return h / (np.abs(h).max() + 1e-8)


def make_baseband(mod_type: int, pulse_len: int, mod_index: float) -> np.ndarray:
    """Returns complex baseband of exact length pulse_len."""
    t = np.linspace(0, 1, pulse_len)
    if mod_type == 0:   # CW
        return np.exp(1j * 2 * np.pi * 0.05 * np.arange(pulse_len))
    elif mod_type == 1:  # LFM
        phase = np.pi * (mod_index * 0.4) * t**2 * pulse_len
        return np.exp(1j * phase)
    elif mod_type == 2:  # BPSK
        n_syms = max(2, int(pulse_len * mod_index / 4))
        sps = max(1, pulse_len // n_syms)
        symbols = np.random.choice([-1, 1], size=n_syms)
        chip = np.repeat(symbols, sps)
        if len(chip) < pulse_len:
            chip = np.pad(chip, (0, pulse_len - len(chip)), mode='edge')
        return chip[:pulse_len].astype(complex)
    else:                # QPSK
        n_syms = max(2, int(pulse_len * mod_index / 4))
        sps = max(1, pulse_len // n_syms)
        consts = np.array([1+1j, -1+1j, -1-1j, 1-1j]) / np.sqrt(2)
        symbols = consts[np.random.randint(0, 4, n_syms)]
        chip = np.repeat(symbols, sps)
        if len(chip) < pulse_len:
            chip = np.pad(chip, (0, pulse_len - len(chip)), mode='edge')
        return chip[:pulse_len]


def generate_pulse(mod_type: int, pulse_duration: float = 0.8,
                   amplitude: float = 0.8, rise_time: float = 0.05,
                   fall_time: float = 0.05, filter_bw: float = 0.3,
                   mod_index: float = 0.5, n: int = N_SAMPLES) -> np.ndarray:
    pulse_len = max(8, int(pulse_duration * n))
    rise_len  = max(1, int(rise_time * pulse_len))
    fall_len  = max(1, int(fall_time * pulse_len))

    baseband = make_baseband(mod_type, pulse_len, mod_index)

    env = np.ones(pulse_len)
    env[:rise_len]  = 0.5 * (1 - np.cos(np.pi * np.arange(rise_len) / rise_len))
    env[-fall_len:] = 0.5 * (1 - np.cos(np.pi * np.arange(fall_len, 0, -1) / fall_len))
    baseband = baseband * env * amplitude

    flen = min(31, pulse_len - 2)
    if flen % 2 == 0: flen -= 1
    flen = max(3, flen)
    h = sinc_filter(flen, filter_bw)
    baseband = np.convolve(baseband, h, mode='same')  # 'same' preserves pulse_len
    baseband += 0.02 * (np.random.randn(pulse_len) + 1j * np.random.randn(pulse_len))

    full  = np.zeros(n, dtype=complex)
    start = (n - pulse_len) // 2
    full[start:start + pulse_len] = baseband

    iq = np.empty(n, dtype=np.float32)
    iq[0::2] = full.real[:n//2]
    iq[1::2] = full.imag[:n//2]
    return iq


def generate_synthetic_dataset(n_samples: int = N_SYNTHETIC):
    iq_list, mod_list, label_list = [], [], []
    for _ in range(n_samples):
        mod = np.random.randint(0, N_MOD_CLASSES)
        pd  = float(np.random.uniform(0.5, 0.95))
        amp = float(np.random.uniform(0.3, 1.0))
        rt  = float(np.random.uniform(0.02, 0.12))
        ft  = float(np.random.uniform(0.02, 0.12))
        fbw = float(np.random.uniform(0.15, 0.45))
        mi  = float(np.random.uniform(0.2, 0.8))
        iq  = generate_pulse(mod, pd, amp, rt, ft, fbw, mi, n=N_SAMPLES)
        iq_list.append(iq)
        mod_list.append(mod)
        label_list.append([pd, amp, rt, ft, fbw, mi])
    return (np.stack(iq_list).astype(np.float32),
            np.array(mod_list, dtype=np.int64),
            np.array(label_list, dtype=np.float32))

In [ ]:
# ── Dataset torch wrapper ─────────────────────────────────────────────────────

class PulseDataset(Dataset):
    """
    Works for both RadioML and synthetic sources.
    Returns (iq, task, labels) where labels are per-sample physical params.
    For RadioML, synthetic labels are estimated from the signal itself.
    """

    def __init__(self, iq: np.ndarray, mod: np.ndarray,
                 labels: Optional[np.ndarray] = None):
        self.iq  = torch.from_numpy(iq)
        self.mod = torch.from_numpy(mod)
        if labels is None:
            # For RadioML: estimate a few observable params from the signal
            labels = self._estimate_labels(iq, mod)
        self.labels = torch.from_numpy(labels.astype(np.float32))

    @staticmethod
    def _estimate_labels(iq: np.ndarray, mod: np.ndarray) -> np.ndarray:
        """Estimate observable parameters from the IQ signal for RadioML bursts."""
        n = len(iq)
        labels = np.zeros((n, 6), dtype=np.float32)
        for i, s in enumerate(iq):
            I, Q = s[0::2].astype(float), s[1::2].astype(float)
            env  = np.sqrt(I**2 + Q**2)
            # pulse_duration: fraction of samples above 20% of peak envelope
            thresh   = 0.2 * env.max()
            active   = (env > thresh).sum()
            labels[i, 0] = float(active) / len(env)    # pulse_duration proxy
            # amplitude: RMS
            labels[i, 1] = float(np.sqrt(np.mean(env**2)))
            # rise/fall: fraction of pulse from 0→peak, peak→0 (rough)
            labels[i, 2] = 0.05   # default; no ground truth in RadioML
            labels[i, 3] = 0.05
            # filter_bw: spectral centroid bandwidth estimate
            spec  = np.abs(np.fft.fft(I + 1j*Q))**2
            spec  = spec[:len(spec)//2]
            freqs = np.linspace(0, 0.5, len(spec))
            spec_n = spec / (spec.sum() + 1e-10)
            bw    = float(np.sqrt(np.sum(spec_n * (freqs - np.sum(spec_n*freqs))**2)))
            labels[i, 4] = float(np.clip(bw * 4, 0.05, 0.5))
            # mod_index: normalised variance of instantaneous freq
            phase      = np.unwrap(np.angle(I + 1j*Q))
            inst_freq  = np.diff(phase) / (2 * np.pi)
            labels[i, 5] = float(np.clip(np.std(inst_freq) * 10, 0, 1))
        return labels

    def __len__(self):  return len(self.iq)
    def __getitem__(self, idx):
        return self.iq[idx], self.mod[idx], self.labels[idx]


# ── Load / build dataset ──────────────────────────────────────────────────────
print('Loading dataset …')
if USE_RADIOML:
    ok = download_radioml(RADIOML_PATH)
    if ok:
        iq_all, mod_all, snr_all = load_radioml(RADIOML_PATH)
        dataset = PulseDataset(iq_all, mod_all)   # labels estimated from signal
        SOURCE  = 'RadioML 2016.10A'
    else:
        print('  Falling back to synthetic generator.')
        USE_RADIOML = False

if not USE_RADIOML:
    iq_all, mod_all, label_all = generate_synthetic_dataset(N_SYNTHETIC)
    dataset = PulseDataset(iq_all, mod_all, label_all)
    SOURCE  = 'Synthetic'

n_train = int(0.85 * len(dataset))
n_val   = len(dataset) - n_train
train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f'Source: {SOURCE} | Train: {n_train:,} | Val: {n_val:,} | Batches/epoch: {len(train_dl)}')

In [ ]:
# ── Visualise a batch ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
iq_b, mod_b, _ = next(iter(train_dl))
iq_b = iq_b.numpy();  mod_b = mod_b.numpy()

for col, cls in enumerate(range(N_MOD_CLASSES)):
    idxs = np.where(mod_b == cls)[0]
    if len(idxs) == 0:
        axes[0, col].set_title(MOD_NAMES[cls] + ' (no sample)')
        continue
    s = iq_b[idxs[0]]
    I, Q = s[0::2], s[1::2]
    axes[0, col].plot(I, lw=0.8, color='steelblue', label='I')
    axes[0, col].plot(Q, lw=0.8, color='coral', alpha=0.8, label='Q')
    axes[0, col].set_title(MOD_NAMES[cls]); axes[0, col].legend(fontsize=8)
    spec = np.abs(np.fft.fftshift(np.fft.fft(I + 1j*Q)))**2
    axes[1, col].plot(10*np.log10(spec + 1e-12), lw=0.8, color='teal')
    axes[1, col].set_ylabel('dB' if col == 0 else '')

axes[0, 0].set_ylabel('Amplitude')
fig.suptitle(f'Training examples — {SOURCE}  (row 1: I/Q  row 2: power spectrum)', fontsize=11)
plt.tight_layout(); plt.show()

## 2. Model components

In [ ]:
# ── Complex-valued linear layer (Trabelsi et al. 2018) ────────────────────────
# W = W_r + j·W_i,  x = x_r + j·x_i
# out_r = W_r·x_r − W_i·x_i
# out_i = W_r·x_i + W_i·x_r

class ComplexLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.W_r = nn.Linear(in_features, out_features, bias=True)
        self.W_i = nn.Linear(in_features, out_features, bias=False)
        sigma = 1.0 / np.sqrt(2 * in_features)
        nn.init.normal_(self.W_r.weight, 0, sigma)
        nn.init.normal_(self.W_i.weight, 0, sigma)

    def forward(self, x_r, x_i):
        return self.W_r(x_r) - self.W_i(x_i), self.W_r(x_i) + self.W_i(x_r)


class CVNNDecoder(nn.Module):
    """Complex-valued MLP decoder for filter_shape latent slot."""
    def __init__(self, in_dim: int, filter_len: int = 32, hidden: int = 32):
        super().__init__()
        self.filter_len = filter_len
        self.embed = nn.Linear(in_dim, hidden * 2)   # lift to complex space
        self.cv1   = ComplexLinear(hidden, hidden)
        self.cv2   = ComplexLinear(hidden, filter_len)
        self.act   = nn.LeakyReLU(0.1)

    def forward(self, z: torch.Tensor):
        h    = self.embed(z)
        x_r  = self.act(h[..., :h.shape[-1]//2])
        x_i  = self.act(h[..., h.shape[-1]//2:])
        x_r, x_i = self.cv1(x_r, x_i)
        x_r, x_i = self.act(x_r), self.act(x_i)
        x_r, x_i = self.cv2(x_r, x_i)
        return torch.stack([x_r, x_i], dim=-1).flatten(-2)  # (B, filter_len*2)


def gumbel_softmax_sample(logits: torch.Tensor, tau: float = 1.0):
    gumbels = -torch.empty_like(logits).exponential_().log()
    return F.softmax((logits + gumbels) / max(tau, 0.05), dim=-1)

In [ ]:
# ── Named latent container ────────────────────────────────────────────────────

@dataclass
class NamedLatent:
    mu_pulse_dur: torch.Tensor; lv_pulse_dur: torch.Tensor
    mu_mod_type:  torch.Tensor; lv_mod_type:  torch.Tensor
    mu_mod_cont:  torch.Tensor; lv_mod_cont:  torch.Tensor
    mu_filter:    torch.Tensor; lv_filter:    torch.Tensor
    mu_rise:      torch.Tensor; lv_rise:      torch.Tensor
    mu_fall:      torch.Tensor; lv_fall:      torch.Tensor
    mu_amplitude: torch.Tensor; lv_amplitude: torch.Tensor
    mu_residuals: torch.Tensor; lv_residuals: torch.Tensor

    z_pulse_dur:  Optional[torch.Tensor] = field(default=None, repr=False)
    z_mod_type:   Optional[torch.Tensor] = field(default=None, repr=False)
    z_mod_cont:   Optional[torch.Tensor] = field(default=None, repr=False)
    z_filter:     Optional[torch.Tensor] = field(default=None, repr=False)
    z_rise:       Optional[torch.Tensor] = field(default=None, repr=False)
    z_fall:       Optional[torch.Tensor] = field(default=None, repr=False)
    z_amplitude:  Optional[torch.Tensor] = field(default=None, repr=False)
    z_residuals:  Optional[torch.Tensor] = field(default=None, repr=False)

    def reparameterize(self, tau: float = 1.0):
        def g(mu, lv):
            return mu + torch.exp(0.5 * lv) * torch.randn_like(lv)
        self.z_pulse_dur = g(self.mu_pulse_dur, self.lv_pulse_dur)
        self.z_mod_type  = gumbel_softmax_sample(self.mu_mod_type, tau)
        self.z_mod_cont  = g(self.mu_mod_cont,  self.lv_mod_cont)
        self.z_filter    = g(self.mu_filter,    self.lv_filter)
        self.z_rise      = g(self.mu_rise,      self.lv_rise)
        self.z_fall      = g(self.mu_fall,      self.lv_fall)
        self.z_amplitude = g(self.mu_amplitude, self.lv_amplitude)
        self.z_residuals = g(self.mu_residuals, self.lv_residuals)

    def z_concat(self):
        return torch.cat([self.z_pulse_dur, self.z_mod_type, self.z_mod_cont,
                          self.z_filter, self.z_rise, self.z_fall,
                          self.z_amplitude, self.z_residuals], dim=-1)

    def kl_loss(self):
        def kl(mu, lv):
            return -0.5 * (1 + lv - mu.pow(2) - lv.exp()).sum(dim=-1).mean()
        return sum(kl(getattr(self, f'mu_{s}'), getattr(self, f'lv_{s}'))
                   for s in ['pulse_dur','mod_type','mod_cont','filter',
                              'rise','fall','amplitude','residuals'])

In [ ]:
# ── Encoder ───────────────────────────────────────────────────────────────────

class Encoder(nn.Module):
    def __init__(self, n_tasks: int = N_MOD_CLASSES, task_emb_dim: int = 16):
        super().__init__()
        self.task_emb = nn.Embedding(n_tasks, task_emb_dim)
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32,  7, stride=2, padding=3), nn.GELU(),   # → 64
            nn.Conv1d(32, 64, 5, stride=2, padding=2), nn.GELU(),   # → 32
            nn.Conv1d(64, 128,3, stride=2, padding=1), nn.GELU(),   # → 16
            nn.Conv1d(128,128,3, stride=2, padding=1), nn.GELU(),   # → 8
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
        )
        self.shared = nn.Sequential(
            nn.Linear(128 + task_emb_dim, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, 256), nn.GELU(),
        )
        def head(d): return nn.Linear(256, d * 2)
        self.h_pd = head(DIM_PULSE_DUR);  self.h_mt = head(DIM_MOD_TYPE)
        self.h_mc = head(DIM_MOD_CONT);   self.h_fi = head(DIM_FILTER)
        self.h_ri = head(DIM_RISE);       self.h_fa = head(DIM_FALL)
        self.h_am = head(DIM_AMPLITUDE);  self.h_re = head(DIM_RESIDUALS)

    def forward(self, iq, task):
        h = self.shared(torch.cat([self.cnn(iq.unsqueeze(1)), self.task_emb(task)], -1))
        def sp(x):
            m = x.shape[-1]//2; return x[...,:m], x[...,m:]
        pd=sp(self.h_pd(h)); mt=sp(self.h_mt(h)); mc=sp(self.h_mc(h)); fi=sp(self.h_fi(h))
        ri=sp(self.h_ri(h)); fa=sp(self.h_fa(h)); am=sp(self.h_am(h)); re=sp(self.h_re(h))
        return NamedLatent(*pd, *mt, *mc, *fi, *ri, *fa, *am, *re)


# ── Bespoke decoders (the forcing functions) ──────────────────────────────────

class ScalarDecoder(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,32),nn.GELU(),nn.Linear(32,16),nn.GELU(),
                                 nn.Linear(16,1),nn.Sigmoid())
    def forward(self, z): return self.net(z).squeeze(-1)


class VectorDecoder(nn.Module):
    def __init__(self, in_d, out_d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_d,64),nn.GELU(),nn.Linear(64,out_d))
    def forward(self, z): return self.net(z)


class IQReconDecoder(nn.Module):
    def __init__(self, z_dim, n_out=N_SAMPLES):
        super().__init__()
        self.n_out = n_out
        self.proj  = nn.Sequential(nn.Linear(z_dim,256),nn.GELU(),nn.Linear(256,128*4))
        self.deconv = nn.Sequential(
            nn.ConvTranspose1d(128,128,4,stride=2,padding=1),nn.GELU(),
            nn.ConvTranspose1d(128, 64,4,stride=2,padding=1),nn.GELU(),
            nn.ConvTranspose1d( 64, 32,4,stride=2,padding=1),nn.GELU(),
            nn.ConvTranspose1d( 32, 16,4,stride=2,padding=1),nn.GELU(),
            nn.ConvTranspose1d( 16,  1,4,stride=2,padding=1),nn.Tanh(),
        )
    def forward(self, z):
        h = self.proj(z).view(-1, 128, 4)
        h = self.deconv(h).squeeze(1)
        if h.shape[-1] != self.n_out:
            h = F.interpolate(h.unsqueeze(1), self.n_out, mode='linear',
                              align_corners=False).squeeze(1)
        return h

In [ ]:
# ── Full model ────────────────────────────────────────────────────────────────

class TCVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder       = Encoder()
        self.dec_pulse_dur = ScalarDecoder(DIM_PULSE_DUR)
        self.dec_amplitude = ScalarDecoder(DIM_AMPLITUDE)
        self.dec_rise      = ScalarDecoder(DIM_RISE)
        self.dec_fall      = ScalarDecoder(DIM_FALL)
        self.dec_mod_cont  = VectorDecoder(DIM_MOD_CONT, DIM_MOD_CONT)
        self.dec_filter    = CVNNDecoder(DIM_FILTER, filter_len=32)
        self.dec_iq        = IQReconDecoder(DIM_TOTAL)
        self.gumbel_tau    = 1.0

    def forward(self, iq, task):
        lat = self.encoder(iq, task)
        lat.reparameterize(self.gumbel_tau)
        return dict(
            latent        = lat,
            recon_iq      = self.dec_iq(lat.z_concat()),
            pred_pulse_dur= self.dec_pulse_dur(lat.z_pulse_dur),
            pred_amplitude= self.dec_amplitude(lat.z_amplitude),
            pred_rise     = self.dec_rise(lat.z_rise),
            pred_fall     = self.dec_fall(lat.z_fall),
            pred_mod_cont = self.dec_mod_cont(lat.z_mod_cont),
            pred_filter   = self.dec_filter(lat.z_filter),
        )


model = TCVAE().to(DEVICE)
n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_p:,}')

# shape smoke-test
with torch.no_grad():
    _iq, _t, _ = next(iter(train_dl))
    _out = model(_iq.to(DEVICE), _t.to(DEVICE))
print(f'recon_iq: {_out["recon_iq"].shape}  pred_filter: {_out["pred_filter"].shape}')

## 3. Loss & training

In [ ]:
def compute_loss(out, iq, labels, task, beta, aux_w):
    lat = out['latent']
    L_recon   = F.mse_loss(out['recon_iq'], iq)
    L_kl      = lat.kl_loss()
    L_pd      = F.mse_loss(out['pred_pulse_dur'], labels[:,0])
    L_amp     = F.mse_loss(out['pred_amplitude'], labels[:,1])
    L_rise    = F.mse_loss(out['pred_rise'],      labels[:,2])
    L_fall    = F.mse_loss(out['pred_fall'],       labels[:,3])
    L_modtype = F.cross_entropy(lat.mu_mod_type, task)
    target_mc = labels[:,5:6].expand_as(out['pred_mod_cont'])
    L_modcont = 1.0 - F.cosine_similarity(out['pred_mod_cont'], target_mc, dim=-1).mean()

    # filter target: windowed sinc based on estimated bandwidth
    with torch.no_grad():
        bw_np = labels[:,4].cpu().numpy()
        tf = np.zeros((len(bw_np), 64), dtype=np.float32)
        for i, bw in enumerate(bw_np):
            h = sinc_filter(32, float(bw))
            tf[i,0::2] = h
        target_filt = torch.from_numpy(tf).to(iq.device)
    L_filter = F.mse_loss(out['pred_filter'], target_filt)

    L_aux   = L_pd + L_amp + L_rise + L_fall + L_modtype + L_modcont + L_filter
    L_total = L_recon + beta * L_kl + aux_w * L_aux
    return dict(total=L_total, recon=L_recon.item(), kl=L_kl.item(),
                pulse_dur=L_pd.item(), amplitude=L_amp.item(),
                mod_type=L_modtype.item(), filter=L_filter.item())


optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-5)

def beta(e):     return min(BETA_MAX, BETA_MAX * e / max(1, WARMUP_EPOCHS))
def aux_w(e):    return min(1.0, max(0.0, (e - AUX_START) / 20.0))
def tau(e):      return max(0.2, 1.0 - 0.8 * e / N_EPOCHS)

def run_epoch(loader, train=True, **kw):
    model.train(train)
    tot = {k: 0.0 for k in ['total','recon','kl','pulse_dur','amplitude','mod_type','filter']}
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for iq, task, labels in loader:
            iq, task, labels = iq.to(DEVICE), task.to(DEVICE), labels.to(DEVICE)
            if train: optimizer.zero_grad()
            out = model(iq, task)
            ls  = compute_loss(out, iq, labels, task, **kw)
            if train:
                ls['total'].backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            for k in tot:
                v = ls[k]; tot[k] += v.item() if hasattr(v,'item') else v
    n = len(loader)
    return {k: v/n for k, v in tot.items()}


history = {k:[] for k in ['train','val','recon','kl','pulse_dur','amplitude','mod_type','filter']}
print(f'\nTraining {N_EPOCHS} epochs on {SOURCE} ({DEVICE})\n')
print(f'{"Ep":>4}  {"β":>5}  {"α":>5}  {"τ":>5}  {"train":>8}  {"val":>8}  {"recon":>8}  {"mod_ce":>8}')

for ep in range(1, N_EPOCHS+1):
    model.gumbel_tau = tau(ep)
    kw = dict(beta=beta(ep), aux_w=aux_w(ep))
    tr = run_epoch(train_dl, train=True,  **kw)
    va = run_epoch(val_dl,   train=False, **kw)
    scheduler.step()
    for k in ['recon','kl','pulse_dur','amplitude','mod_type','filter']:
        history[k].append(tr[k])
    history['train'].append(tr['total']); history['val'].append(va['total'])
    if ep % 5 == 0 or ep == 1:
        print(f'{ep:>4}  {beta(ep):>5.3f}  {aux_w(ep):>5.3f}  {tau(ep):>5.3f}  '
              f'{tr["total"]:>8.4f}  {va["total"]:>8.4f}  '
              f'{tr["recon"]:>8.4f}  {tr["mod_type"]:>8.4f}')

print('Done.')

## 4. Training curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
eps = range(1, N_EPOCHS+1)
plots = [
    (axes[0,0], 'Total loss',          [('Train', history['train']),('Val', history['val'])]),
    (axes[0,1], 'Reconstruction MSE',  [('Recon', history['recon'])]),
    (axes[0,2], 'KL divergence',       [('KL', history['kl'])]),
    (axes[1,0], 'Scalar aux losses',   [('pulse_dur', history['pulse_dur']),('amplitude',history['amplitude'])]),
    (axes[1,1], 'mod_type cross-entropy', [('CE', history['mod_type'])]),
    (axes[1,2], 'Filter MSE (CVNN)',   [('filter', history['filter'])]),
]
for ax, title, series in plots:
    for label, data in series:
        ax.plot(eps, data, label=label, lw=1.2)
    ax.axvline(WARMUP_EPOCHS, color='gray', ls='--', alpha=0.4, lw=0.8)
    ax.axvline(AUX_START,     color='teal', ls=':',  alpha=0.5, lw=0.8)
    ax.set_title(title, fontsize=10); ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)

fig.suptitle(f'TC-VAE training — {SOURCE}  (gray: KL warmup end, teal: aux loss start)', fontsize=11)
plt.tight_layout(); plt.show()

## 5. Evaluation — latent alignment

In [ ]:
# Collect μ values over full val set
all_mu_pd, all_mu_amp, all_mu_mt = [], [], []
all_gt_pd, all_gt_amp, all_gt_mod = [], [], []
model.eval()
with torch.no_grad():
    for iq_b, task_b, lab_b in val_dl:
        lat = model.encoder(iq_b.to(DEVICE), task_b.to(DEVICE))
        all_mu_pd.append(lat.mu_pulse_dur.cpu().numpy())
        all_mu_amp.append(lat.mu_amplitude.cpu().numpy())
        all_mu_mt.append(lat.mu_mod_type.cpu().numpy())
        all_gt_pd.append(lab_b[:,0].numpy())
        all_gt_amp.append(lab_b[:,1].numpy())
        all_gt_mod.append(task_b.numpy())

mu_pd  = np.concatenate(all_mu_pd).squeeze()
mu_amp = np.concatenate(all_mu_amp).squeeze()
mu_mt  = np.concatenate(all_mu_mt)
gt_pd  = np.concatenate(all_gt_pd)
gt_amp = np.concatenate(all_gt_amp)
gt_mod = np.concatenate(all_gt_mod)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (mu, gt, name) in zip(axes[:2], [
    (mu_pd, gt_pd, 'pulse_duration'), (mu_amp, gt_amp, 'amplitude')]):
    ax.scatter(gt, mu, s=3, alpha=0.2, rasterized=True, color='steelblue')
    r = np.corrcoef(gt, mu)[0,1]
    ax.set_xlabel(f'GT {name}'); ax.set_ylabel('Latent μ')
    ax.set_title(f'{name}\nr = {r:.3f}')

# mod_type confusion
pred_mod = np.argmax(mu_mt, axis=1)
conf = np.zeros((4,4), dtype=int)
for g,p in zip(gt_mod, pred_mod): conf[g,p] += 1
acc = conf.diagonal().sum() / len(gt_mod)
im = axes[2].imshow(conf, cmap='Blues')
axes[2].set_xticks(range(4)); axes[2].set_xticklabels(MOD_NAMES, rotation=30, ha='right', fontsize=8)
axes[2].set_yticks(range(4)); axes[2].set_yticklabels(MOD_NAMES, fontsize=8)
axes[2].set_title(f'mod_type confusion  acc={acc:.3f}')
for i in range(4):
    for j in range(4): axes[2].text(j, i, conf[i,j], ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=axes[2])

# IQ reconstruction quality: scatter original vs recon amplitude
with torch.no_grad():
    iq_b, task_b, _ = next(iter(val_dl))
    out = model(iq_b.to(DEVICE), task_b.to(DEVICE))
orig_rms  = iq_b.numpy().std(axis=1)
recon_rms = out['recon_iq'].cpu().numpy().std(axis=1)
axes[3].scatter(orig_rms, recon_rms, s=4, alpha=0.3, color='coral')
lim = max(orig_rms.max(), recon_rms.max())
axes[3].plot([0,lim],[0,lim],'k--',lw=0.8, alpha=0.5)
r = np.corrcoef(orig_rms, recon_rms)[0,1]
axes[3].set_xlabel('Original RMS'); axes[3].set_ylabel('Recon RMS')
axes[3].set_title(f'IQ recon quality  r={r:.3f}')

plt.suptitle(f'Named latent alignment — {SOURCE}', fontsize=11)
plt.tight_layout(); plt.show()
print(f'\nmod_type accuracy: {acc:.1%}')

## 6. IQ reconstruction gallery

In [ ]:
model.eval()
iq_b, task_b, _ = next(iter(val_dl))
with torch.no_grad():
    out = model(iq_b.to(DEVICE), task_b.to(DEVICE))
iq_orig  = iq_b.numpy()
iq_recon = out['recon_iq'].cpu().numpy()
tasks    = task_b.numpy()

fig, axes = plt.subplots(4, 4, figsize=(16, 10))
for row, cls in enumerate(range(4)):
    idxs = np.where(tasks == cls)[0]
    if not len(idxs): continue
    i = idxs[0]
    for col, (arr, style, title) in enumerate([
        (iq_orig[i],  '-',  f'{MOD_NAMES[cls]} orig I'),
        (iq_orig[i],  '-',  'orig Q'),
        (iq_recon[i], '--', 'recon I'),
        (iq_recon[i], '--', 'recon Q'),
    ]):
        component = arr[0::2] if 'I' in title else arr[1::2]
        color = 'steelblue' if 'I' in title else 'coral'
        axes[row,col].plot(component, lw=0.9, color=color, linestyle=style)
        axes[row,col].set_title(title, fontsize=9)
        axes[row,col].set_xticks([])

fig.suptitle('IQ reconstruction — solid: original  dashed: reconstructed', fontsize=11)
plt.tight_layout(); plt.show()

## 7. Latent traversal — causal verification

In [ ]:
def traverse(iq_ex, task_ex, slot: str, n_steps=7, val_range=(-2.5, 2.5)):
    """
    Vary one named latent dim linearly while holding all others at posterior μ.
    This is the causal check: does manipulating z.amplitude actually change amplitude?
    """
    model.eval()
    with torch.no_grad():
        lat = model.encoder(iq_ex.unsqueeze(0).to(DEVICE), task_ex.unsqueeze(0).to(DEVICE))

    # Build z_concat with all slots at their μ, varying target slot
    recons = []
    slots = ['pulse_dur','mod_type','mod_cont','filter','rise','fall','amplitude','residuals']
    for v in np.linspace(*val_range, n_steps):
        zs = []
        for s in slots:
            mu = getattr(lat, f'mu_{s}').clone()
            if s == slot:
                mu = torch.full_like(mu, v)
            if s == 'mod_type':
                zs.append(F.softmax(mu, dim=-1))
            else:
                zs.append(mu)
        z = torch.cat(zs, dim=-1)
        with torch.no_grad():
            r = model.dec_iq(z).cpu().numpy().squeeze()
        recons.append(r)

    values = np.linspace(*val_range, n_steps)
    fig, axes = plt.subplots(1, n_steps, figsize=(3*n_steps, 3), sharey=True)
    for ax, r, v in zip(axes, recons, values):
        ax.plot(r[0::2], lw=0.8, color='steelblue')
        ax.plot(r[1::2], lw=0.8, color='coral', alpha=0.75)
        ax.set_title(f'{v:.1f}', fontsize=9); ax.set_xticks([])
    axes[0].set_ylabel('IQ amplitude')
    fig.suptitle(f'z_{slot} traversal — {MOD_NAMES[task_ex.item()]}', fontsize=10)
    plt.tight_layout(); plt.show()


# Run traversals for each modulation class
iq_b, task_b, _ = next(iter(val_dl))
for cls in range(N_MOD_CLASSES):
    idxs = (task_b == cls).nonzero(as_tuple=True)[0]
    if len(idxs) == 0: continue
    traverse(iq_b[idxs[0]], task_b[idxs[0]], 'amplitude')

# Also traverse filter_shape for class 0
idxs = (task_b == 0).nonzero(as_tuple=True)[0]
if len(idxs): traverse(iq_b[idxs[0]], task_b[idxs[0]], 'filter')

## 8. CVNN filter decoder — inspect complex filter coefficients

In [ ]:
model.eval()
iq_b, task_b, lab_b = next(iter(val_dl))
with torch.no_grad():
    lat = model.encoder(iq_b.to(DEVICE), task_b.to(DEVICE))
    lat.reparameterize(tau=0.2)
    pred_f = model.dec_filter(lat.z_filter).cpu().numpy()

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for col in range(4):
    bw  = float(lab_b[col, 4])
    gt  = sinc_filter(32, bw)
    pr  = pred_f[col]
    pr_r, pr_i = pr[0::2], pr[1::2]

    axes[0,col].plot(gt, lw=1.2, color='steelblue', label='GT real')
    axes[0,col].plot(pr_r, lw=1.0, color='coral', ls='--', label='Pred real')
    axes[0,col].plot(pr_i, lw=0.8, color='green', ls=':', alpha=0.7, label='Pred imag')
    axes[0,col].set_title(f'BW={bw:.2f}'); axes[0,col].legend(fontsize=7)

    H_gt = np.abs(np.fft.fft(gt, 128)[:64])
    H_pr = np.abs(np.fft.fft(pr_r + 1j*pr_i, 128)[:64])
    axes[1,col].plot(H_gt, lw=1.2, color='steelblue', label='GT')
    axes[1,col].plot(H_pr, lw=1.0, color='coral', ls='--', label='Pred')
    axes[1,col].set_title('Freq response'); axes[1,col].legend(fontsize=7)

fig.suptitle('CVNN filter decoder — ground truth vs predicted complex filter', fontsize=11)
plt.tight_layout(); plt.show()

## 9. Save checkpoint

In [ ]:
ckpt = dict(model_state=model.state_dict(),
            optimizer_state=optimizer.state_dict(),
            history=history, epochs=N_EPOCHS, source=SOURCE)
torch.save(ckpt, 'tc_vae_checkpoint.pt')
print('Saved tc_vae_checkpoint.pt')

# To reload:
# ckpt = torch.load('tc_vae_checkpoint.pt', map_location=DEVICE)
# model.load_state_dict(ckpt['model_state'])